# Chapter 6: How Neural Networks Learn

Neural networks are inspired by the brain — but simplified heavily.

A biological neuron:
1. Receives signals from other neurons
2. If total signal is strong enough → fires
3. Sends signal to the next neurons

An artificial neuron does the same:
1. Receives numbers (inputs × weights)
2. Sums them up, applies an activation function
3. Outputs a number to the next layer

---
## A Single Neuron

The simplest neural network: one neuron.

```
input1 ──×w1──┐
               ├── sum + bias ── activation ── output
input2 ──×w2──┘
```

```
output = activation(input1×w1 + input2×w2 + bias)
```

- **Weights**: how important each input is
- **Bias**: shifts the decision boundary (like the intercept in `y = mx + b`)
- **Activation**: decides whether the neuron "fires"

In [ ]:
import numpy as np

# A single neuron
def neuron(inputs, weights, bias):
    weighted_sum = sum(i * w for i, w in zip(inputs, weights)) + bias
    return weighted_sum

# Example: should I go outside?
# inputs: [temperature (normalized), is_sunny]
# weights: how much each factor matters
weights = [0.7, 0.3]   # temperature matters more
bias = -0.5             # slightly reluctant to go out

scenarios = [
    ([0.2, 0.0], "cold, cloudy"),
    ([0.8, 1.0], "warm, sunny"),
    ([0.5, 0.0], "mild, cloudy"),
    ([0.9, 1.0], "hot, sunny"),
]

print("Single Neuron (no activation):")
print(f"{'Scenario':<16} {'Inputs':<14} {'Weighted Sum':>12} {'Go out?':>8}")
print(f"{'─'*16} {'─'*14} {'─'*12} {'─'*8}")
for inputs, desc in scenarios:
    result = neuron(inputs, weights, bias)
    decision = "Yes" if result > 0 else "No"
    print(f"{desc:<16} {str(inputs):<14} {result:>12.2f} {decision:>8}")

print("\nThe neuron is basically: weighted vote + threshold.")

---
## Activation Functions: The Non-Linear Magic

Without activation functions, stacking layers is pointless — multiple linear operations collapse into one linear operation.

Activation functions add **non-linearity**, letting the network learn complex patterns.

```
Linear only:     layer2(layer1(x)) = just another line
With activation: layer2(σ(layer1(x))) = can learn curves!
```

In [ ]:
# Common activation functions
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def tanh(x):
    return np.tanh(x)

# Visualize them with ASCII
test_values = np.arange(-4, 4.5, 0.5)

activations = {
    "Sigmoid": sigmoid,
    "ReLU": relu,
    "Tanh": tanh,
}

for name, func in activations.items():
    print(f"\n{name}:")
    print(f"  {'Input':>6} {'Output':>8}  Graph")
    for x in test_values:
        y = func(x)
        bar_pos = int((y + 1) * 15) if name == "Tanh" else int(y * 20)
        bar_pos = max(0, min(30, bar_pos))
        bar = " " * bar_pos + "●"
        print(f"  {x:>6.1f} {y:>8.3f}  │{bar}")

In [ ]:
# When to use which?
print("Activation Function Cheat Sheet:")
print()
print(f"{'Function':<10} {'Range':<14} {'Use Case':<30} {'Pros/Cons'}")
print(f"{'─'*10} {'─'*14} {'─'*30} {'─'*30}")
print(f"{'Sigmoid':<10} {'(0, 1)':<14} {'Output layer (probability)':<30} {'Can saturate (vanishing gradient)'}")
print(f"{'Tanh':<10} {'(-1, 1)':<14} {'Hidden layers (older nets)':<30} {'Zero-centered, but still saturates'}")
print(f"{'ReLU':<10} {'[0, ∞)':<14} {'Hidden layers (modern default)':<30} {'Fast, simple, can "die" (output 0)'}")
print()
print("Modern standard: ReLU for hidden layers, sigmoid/softmax for output.")

---
## Layers: Stacking Neurons

A single neuron can only learn linear boundaries (straight lines).
Multiple neurons in **layers** can learn any pattern.

```
Input Layer    Hidden Layer(s)    Output Layer
  [x1] ────→  [h1] ─────────→  [output]
  [x2] ────→  [h2] ─────────→
              [h3]
```

- **Input layer**: your raw data (features)
- **Hidden layers**: where patterns are discovered
- **Output layer**: the final prediction

Each connection has its own weight. The network learns ALL weights simultaneously.

In [ ]:
# Build a network layer by layer
def dense_layer(inputs, weights, biases, activation=sigmoid):
    """One fully-connected layer."""
    z = inputs @ weights + biases
    return activation(z)

# Network: 2 inputs → 3 hidden → 1 output
np.random.seed(42)
W1 = np.random.randn(2, 3) * 0.5    # 2 inputs → 3 hidden neurons
b1 = np.zeros((1, 3))
W2 = np.random.randn(3, 1) * 0.5    # 3 hidden → 1 output
b2 = np.zeros((1, 1))

print("Network Architecture: 2 → 3 → 1")
print(f"  W1 shape: {W1.shape} (2 inputs × 3 hidden neurons = 6 weights)")
print(f"  W2 shape: {W2.shape} (3 hidden × 1 output = 3 weights)")
print(f"  Total weights: {W1.size + b1.size + W2.size + b2.size}")
print()

# Forward pass
X = np.array([[1.0, 0.5]])
print(f"Input: {X.flatten()}")

hidden = dense_layer(X, W1, b1, relu)
print(f"\nHidden layer output (3 neurons): {hidden.flatten().round(4)}")
print(f"  Neuron 1: {hidden[0,0]:.4f} (this pattern detected)")
print(f"  Neuron 2: {hidden[0,1]:.4f} (this pattern detected)")
print(f"  Neuron 3: {hidden[0,2]:.4f} (ReLU killed this → not relevant)")

output = dense_layer(hidden, W2, b2, sigmoid)
print(f"\nOutput: {output[0,0]:.4f} (probability)")

---
## Why Do We Need Hidden Layers?

A network with no hidden layers can only draw **straight lines**.

Hidden layers let it draw **curves and complex boundaries**.

Classic example: XOR cannot be solved with a straight line.

In [ ]:
# Single neuron (no hidden layer) vs network (with hidden layer)
# on the XOR problem

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0])

# Can a single neuron solve XOR?
print("Can a single neuron (no hidden layer) solve XOR?\n")

# Try all weight combinations
best_acc = 0
for w1 in np.arange(-2, 2.1, 0.5):
    for w2 in np.arange(-2, 2.1, 0.5):
        for b in np.arange(-2, 2.1, 0.5):
            preds = (sigmoid(X @ np.array([w1, w2]) + b) > 0.5).astype(int)
            acc = np.mean(preds == y)
            if acc > best_acc:
                best_acc = acc

print(f"  Best accuracy with single neuron: {best_acc:.0%}")
print(f"  It can get at most 3/4 correct — XOR needs a curve!")
print()
print("  [0,0]→0  [0,1]→1")
print("  [1,0]→1  [1,1]→0")
print()
print("  No straight line can separate 0s from 1s here.")
print("  Add a hidden layer → problem solved (see Ch 7).")

---
## How Learning Works: The Big Picture

```
1. Initialize weights randomly
2. Forward pass:  input → prediction
3. Loss function: how wrong is the prediction?
4. Backward pass:  calculate gradients (how to fix each weight)
5. Update weights: nudge in the right direction
6. Repeat 2-5 thousands of times
```

Steps 4-5 are called **backpropagation + gradient descent** (covered in detail in Ch 7).

In [ ]:
# Loss functions: measuring "how wrong"
print("Common Loss Functions:\n")

# MSE for regression
print("1. Mean Squared Error (MSE) — for regression")
actual = np.array([3.0, 5.0, 7.0])
predicted = np.array([2.8, 5.3, 6.5])
mse = np.mean((actual - predicted) ** 2)
print(f"   Actual:    {actual}")
print(f"   Predicted: {predicted}")
print(f"   MSE = mean of ({actual - predicted})² = {mse:.4f}")

print()

# Binary cross-entropy for classification
print("2. Binary Cross-Entropy — for classification")
actual_cls = np.array([1, 0, 1, 1])
predicted_prob = np.array([0.9, 0.1, 0.8, 0.6])
bce = -np.mean(actual_cls * np.log(predicted_prob) + (1 - actual_cls) * np.log(1 - predicted_prob))
print(f"   Actual:    {actual_cls}")
print(f"   Predicted: {predicted_prob}")
print(f"   BCE = {bce:.4f}")
print()
print("   BCE punishes confident WRONG predictions heavily:")
for prob in [0.99, 0.9, 0.5, 0.1, 0.01]:
    loss = -np.log(prob)
    bar = "█" * min(30, int(loss * 6))
    print(f"   Correct answer, predicted {prob:.2f} → loss = {loss:.3f}  {bar}")

---
## Gradient Descent: Walking Downhill

The loss function creates a "landscape" where:
- **High points** = bad weights (high loss)
- **Low points** = good weights (low loss)

Gradient descent: feel the slope under your feet, step downhill.

```
new_weight = old_weight - learning_rate × gradient
```

The **gradient** (derivative) tells you:
- **Direction**: which way is downhill
- **Magnitude**: how steep the slope is

In [ ]:
# Visualize gradient descent on a simple function: loss = (w - 3)²
# The minimum is at w = 3

def loss_fn(w):
    return (w - 3) ** 2

def gradient(w):
    return 2 * (w - 3)

# Start at w = -1, try to find the minimum
w = -1.0
lr = 0.1

print("Gradient Descent: finding the minimum of loss = (w - 3)²")
print(f"Starting at w = {w}, learning rate = {lr}")
print(f"True minimum at w = 3\n")

print(f"{'Step':>5} {'w':>8} {'Loss':>8} {'Gradient':>10} {'Direction':>10}  Path")
print(f"{'─'*5} {'─'*8} {'─'*8} {'─'*10} {'─'*10}  {'─'*30}")

for step in range(15):
    loss = loss_fn(w)
    grad = gradient(w)
    direction = "→ right" if grad < 0 else "← left"
    
    # ASCII visualization: position of w on scale -1 to 5
    pos = int((w - (-1)) / 6 * 30)
    pos = max(0, min(29, pos))
    target = int((3 - (-1)) / 6 * 30)
    line = list("." * 30)
    line[target] = "★"
    line[pos] = "●"
    
    print(f"{step:>5} {w:>8.3f} {loss:>8.3f} {grad:>+10.3f} {direction:>10}  {''.join(line)}")
    
    w = w - lr * grad

print(f"\n● approaches ★ with each step — that's gradient descent!")

---
## Learning Rate: How Big Are the Steps?

The learning rate is critical:
- **Too large**: overshoots, bounces around, may never converge
- **Too small**: takes forever to reach the minimum
- **Just right**: steady convergence

In [ ]:
# Compare different learning rates
print("Effect of Learning Rate:\n")

for lr in [0.01, 0.1, 0.5, 0.9, 1.1]:
    w = -1.0
    history = [w]
    for _ in range(20):
        grad = 2 * (w - 3)
        w = w - lr * grad
        history.append(w)
    
    final_loss = loss_fn(history[-1])
    
    if final_loss > 1000:
        status = "DIVERGED!"
    elif final_loss < 0.01:
        status = "converged ✓"
    else:
        status = "still moving..."
    
    path = ""
    for h in history[:10]:
        if -2 < h < 8:
            path += f"{h:.1f}→"
        else:
            path += "∞!"
            break
    
    print(f"  lr={lr:<4}  final w={history[-1]:>8.3f}  loss={final_loss:>10.3f}  {status}")
    print(f"          path: {path[:-1]}")
    print()

---
## Putting It Together: A Complete Neural Network

Let's train a neural network on a real classification problem from scratch.

In [ ]:
# Complete neural network training from scratch
np.random.seed(42)

# Generate spiral data (2 classes, not linearly separable)
N = 100  # points per class
t = np.linspace(0, 2 * np.pi, N)
X_class0 = np.column_stack([t * np.cos(t) * 0.3, t * np.sin(t) * 0.3]) + np.random.randn(N, 2) * 0.2
X_class1 = np.column_stack([t * np.cos(t + np.pi) * 0.3, t * np.sin(t + np.pi) * 0.3]) + np.random.randn(N, 2) * 0.2

X = np.vstack([X_class0, X_class1])
y = np.array([0]*N + [1]*N).reshape(-1, 1)

# Network: 2 → 16 → 8 → 1
W1 = np.random.randn(2, 16) * 0.3
b1 = np.zeros((1, 16))
W2 = np.random.randn(16, 8) * 0.3
b2 = np.zeros((1, 8))
W3 = np.random.randn(8, 1) * 0.3
b3 = np.zeros((1, 1))

lr = 0.1
print("Training: 2 inputs → 16 hidden → 8 hidden → 1 output")
print(f"Total parameters: {W1.size + b1.size + W2.size + b2.size + W3.size + b3.size}")
print(f"Training samples: {len(X)}\n")

for epoch in range(1001):
    # Forward
    z1 = X @ W1 + b1
    a1 = np.maximum(0, z1)  # ReLU
    z2 = a1 @ W2 + b2
    a2 = np.maximum(0, z2)  # ReLU
    z3 = a2 @ W3 + b3
    a3 = sigmoid(z3)        # Sigmoid output
    
    # Loss
    loss = -np.mean(y * np.log(a3 + 1e-8) + (1 - y) * np.log(1 - a3 + 1e-8))
    accuracy = np.mean((a3 > 0.5) == y)
    
    # Backward
    m = len(X)
    dz3 = a3 - y
    dW3 = a2.T @ dz3 / m
    db3 = np.sum(dz3, axis=0, keepdims=True) / m
    
    dz2 = (dz3 @ W3.T) * (z2 > 0)
    dW2 = a1.T @ dz2 / m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m
    
    dz1 = (dz2 @ W2.T) * (z1 > 0)
    dW1 = X.T @ dz1 / m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m
    
    # Update
    W3 -= lr * dW3; b3 -= lr * db3
    W2 -= lr * dW2; b2 -= lr * db2
    W1 -= lr * dW1; b1 -= lr * db1
    
    if epoch % 200 == 0:
        bar = "█" * int(accuracy * 30)
        print(f"  Epoch {epoch:>5} | Loss: {loss:.4f} | Accuracy: {accuracy:.1%}  {bar}")

print(f"\nFinal accuracy: {accuracy:.1%}")
print("A linear model would get ~50% on spiral data.")
print("The hidden layers learned the non-linear boundary!")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Neuron** | weighted sum + bias + activation |
| **Activation** | Adds non-linearity (ReLU, sigmoid, tanh) |
| **Layers** | Input → Hidden(s) → Output |
| **Hidden layers** | Enable learning complex, non-linear patterns |
| **Loss function** | Measures how wrong the prediction is |
| **Gradient descent** | Follow the slope downhill to minimize loss |
| **Learning rate** | Step size — too big diverges, too small is slow |

**Next chapter**: Backpropagation — the algorithm that calculates gradients for every weight